In [64]:
import json
import re
import sqlite3
import os
import sqlglot
from collections import defaultdict
from sqlglot.expressions import Join
from collections import Counter

In [65]:
SPIDER_DB_DIR = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/database" 
FILE_PATH = "/mnt/storage_C1/igorzwirtes/poster_ic/predictions/predictions_reward_reward_A_1400_teste_1.json"

In [66]:
def normalize_value(v):

    if v is None:
        return None

    if isinstance(v, float):
        return round(v, 6)

    if isinstance(v, str):
        return v.strip().lower()

    return v

def normalize_result(rows):
    normalized = []
    for row in rows:
        normalized.append(tuple(normalize_value(v) for v in row))
    
    as_sorted     = sorted(normalized, key=str)   # ordena as tuplas originais
    as_frozensets = sorted((frozenset(row) for row in normalized), key=str) # ignora ordem de colunas
    
    return as_frozensets, as_sorted

def execute_query(db_id, sql):

    db_path = os.path.join(
        SPIDER_DB_DIR,
        db_id,
        f"{db_id}.sqlite"
    )

    conn = None

    try:
        conn = sqlite3.connect(db_path)

        cursor = conn.cursor()

        cursor.execute(sql)

        rows = cursor.fetchall()

        return normalize_result(rows)

    except Exception:
        return None

    finally:
        if conn:
            conn.close()

In [67]:
def syntax_valid(db_id, sql):

    db_path = os.path.join(
        SPIDER_DB_DIR,
        db_id,
        f"{db_id}.sqlite"
    )

    conn = None

    try:
        conn = sqlite3.connect(db_path)

        cur = conn.cursor()

        cur.execute(sql)

        return True

    except Exception:
        return False

    finally:
        if conn:
            conn.close()

def execution_accuracy(gold, pred, db_id):
    gold_results = execute_query(db_id, gold)
    pred_results = execute_query(db_id, pred)

    if gold_results is None or pred_results is None:
        return False

    gold_fs, gold_s = normalize_result(gold_results)
    pred_fs, pred_s = normalize_result(pred_results)

    # Correto se bater de qualquer forma
    return gold_fs == pred_fs or gold_s == pred_s

def exact_match(gold, pred):

    try:
        gold_ast = sqlglot.parse_one(gold)
        pred_ast = sqlglot.parse_one(pred)

        return gold_ast == pred_ast

    except Exception:
        return False

In [68]:
SEMANTIC_EQUIVALENCES = [
    # NOT IN ≈ EXCEPT com JOIN
    (r"\bnot\s+in\s*\(", "except"),
    # JOIN com WHERE ≈ subquery
    (r"\border\s+by\b.*\blimit\s+1\b", "subquery com min/max"),
]

def are_semantically_equivalent(pred_sql, gold_sql, db_id):
    """Executa ambos e compara — se resultado igual, são equivalentes."""
    gold_res = execute_query(db_id, gold_sql)
    pred_res = execute_query(db_id, pred_sql)
    
    if gold_res is None or pred_res is None:
        return False
    
    gold_fs = sorted(frozenset(row) for row in [tuple(normalize_value(v) for v in r) for r in gold_res])
    pred_fs = sorted(frozenset(row) for row in [tuple(normalize_value(v) for v in r) for r in pred_res])
    
    return gold_fs == pred_fs

In [69]:
def classify_error(pred_sql, gold_sql):

    pred_lower = pred_sql.lower()
    gold_lower = gold_sql.lower()

    if not pred_sql or pred_sql.strip() == "":
        return "empty"

    if "join" in gold_lower and "join" not in pred_lower:
        return "missing_join"

    if "group by" in gold_lower and "group by" not in pred_lower:
        return "missing_group_by"

    if "having" in gold_lower and "having" not in pred_lower:
        return "missing_having"

    if "intersect" in gold_lower and "intersect" not in pred_lower:
        return "missing_intersect"

    if "except" in gold_lower and "except" not in pred_lower:
        return "missing_except"

    where_pos = gold_lower.find("where")

    if where_pos != -1:

        after_where = gold_lower[where_pos:]

        if "select" in after_where:

            pred_after_where = pred_lower[
                pred_lower.find("where"):
            ]

            if "select" not in pred_after_where:
                return "missing_subquery"

    if "order by" in gold_lower and "order by" not in pred_lower:
        return "missing_order_by"

    return "wrong_columns_or_values"

In [70]:
with open(FILE_PATH) as f:
    predictions = json.load(f)

In [71]:
evaluated_predictions = []

for p in predictions:

    syntax_ok = syntax_valid(
        p["db_id"],
        p["predicted"]
    )

    ex_acc = execution_accuracy(
        p["gold"],
        p["predicted"],
        p["db_id"]
    )

    em = exact_match(
        p["gold"],
        p["predicted"]
    )

    error_type = None

    if not ex_acc:

        # Verifica se é equivalência semântica antes de classificar
        if are_semantically_equivalent(p["predicted"], p["gold"], p["db_id"]):
            error_type = "semantic_equivalent"  # não é erro real
        else:
            error_type = classify_error(p["predicted"], p["gold"])

        error_type = classify_error(
            p["predicted"],
            p["gold"]
        )

    evaluated_predictions.append({
        **p,
        "syntax_valid": syntax_ok,
        "execution_accuracy": ex_acc,
        "exact_match": em,
        "error_type": error_type
    })

In [72]:
total = len(evaluated_predictions)

syntax_score = sum(
    p["syntax_valid"]
    for p in evaluated_predictions
) / total

execution_score = sum(
    p["execution_accuracy"]
    for p in evaluated_predictions
) / total

exact_match_score = sum(
    p["exact_match"]
    for p in evaluated_predictions
) / total

print("\n" + "=" * 80)
print("GLOBAL METRICS")
print("=" * 80)

print(f"Total Examples:      {total}")
print(f"Syntax Validity:     {syntax_score:.4f}")
print(f"Execution Accuracy:  {execution_score:.4f}")
print(f"Exact Match (AST):   {exact_match_score:.4f}")


GLOBAL METRICS
Total Examples:      1034
Syntax Validity:     0.9874
Execution Accuracy:  0.7824
Exact Match (AST):   0.2534


In [73]:
invalid_queries = [
    p for p in evaluated_predictions
    if not p["syntax_valid"]
]

print("\n" + "=" * 80)
print("INVALID SQL")
print("=" * 80)

print(f"Invalid Queries: {len(invalid_queries)}")

for p in invalid_queries[:5]:

    print("\nQuestion:")
    print(p["question"])

    print("\nPredicted:")
    print(p["predicted"])

    print("-" * 80)


INVALID SQL
Invalid Queries: 13

Question:
Which model of the car has the minimum horsepower?

Predicted:
SELECT T1.Model FROM model_list AS T1 JOIN car_names AS T2 ON T1.Model  =  T2.Model ORDER BY T2.Horsepower LIMIT 1
--------------------------------------------------------------------------------

Question:
Find the make and production time of the cars that were produced in the earliest year?

Predicted:
SELECT Make ,  YEAR FROM cars_data ORDER BY YEAR ASC LIMIT 1
--------------------------------------------------------------------------------

Question:
What is the maker of the carr produced in the earliest year and what year was it?

Predicted:
SELECT T1.Maker ,  T2.Year FROM car_makers AS T1 JOIN cars_data AS T2 ON T1.Id  =  T2.MakeId ORDER BY T2.Year LIMIT 1
--------------------------------------------------------------------------------

Question:
What is the average edispl for all volvos?

Predicted:
SELECT avg(T1.Edispl) FROM cars_data AS T1 JOIN car_names AS T2 ON T1.MakeI

In [74]:
errors = [
    p for p in evaluated_predictions
    if not p["execution_accuracy"]
]

error_counter = Counter(
    p["error_type"]
    for p in errors
)

print("\n" + "=" * 80)
print("ERROR DISTRIBUTION")
print("=" * 80)

for error_type, count in error_counter.most_common():

    percentage = 100 * count / len(errors)

    print(
        f"{error_type:<30} "
        f"{count:<5} "
        f"({percentage:.2f}%)"
    )


ERROR DISTRIBUTION
wrong_columns_or_values        184   (81.78%)
missing_join                   16    (7.11%)
missing_subquery               11    (4.89%)
missing_group_by               8     (3.56%)
missing_intersect              3     (1.33%)
missing_order_by               2     (0.89%)
missing_except                 1     (0.44%)


In [75]:
grouped_errors = defaultdict(list)

for p in errors:
    grouped_errors[p["error_type"]].append(p)

In [76]:
for error_type, examples in grouped_errors.items():

    print("\n" + "=" * 80)
    print(f"ERROR TYPE: {error_type}")
    print(f"COUNT: {len(examples)}")
    print("=" * 80)

    for i, p in enumerate(examples[:], 1):

        print(f"\nExample {i}")

        print("\nQuestion:")
        print(p["question"])

        print("\nGold SQL:")
        print(p["gold"])

        print("\nPredicted SQL:")
        print(p["predicted"])

        print("-" * 80)


ERROR TYPE: wrong_columns_or_values
COUNT: 184

Example 1

Question:
What is the maximum capacity and the average of all stadiums ?

Gold SQL:
select max(capacity), average from stadium

Predicted SQL:
SELECT max(capacity) ,  avg(capacity) FROM stadium
--------------------------------------------------------------------------------

Example 2

Question:
What is the first name of every student who has a dog but does not have a cat?

Gold SQL:
SELECT T1.fname ,  T1.age FROM student AS T1 JOIN has_pet AS T2 ON T1.stuid  =  T2.stuid JOIN pets AS T3 ON T3.petid  =  T2.petid WHERE T3.pettype  =  'dog' AND T1.stuid NOT IN (SELECT T1.stuid FROM student AS T1 JOIN has_pet AS T2 ON T1.stuid  =  T2.stuid JOIN pets AS T3 ON T3.petid  =  T2.petid WHERE T3.pettype  =  'cat')

Predicted SQL:
SELECT T1.Fname FROM Student AS T1 JOIN Has_Pet AS T2 ON T1.StuID  =  T2.StuID JOIN Pets AS T3 ON T2.PetID  =  T3.PetID WHERE T3.PetType  =  "dog" EXCEPT SELECT T1.Fname FROM Student AS T1 JOIN Has_Pet AS T2 ON 